# Nudging round 2

Concept: DEAP unlikely to have found precisely the best solutions, so take its answers and nudge the coefficients a bit to see if there's an even better answer nearby.

First show that this can result in a better fit (using the "fitness" metric) by manually picking out some coefficients that look better.

Show that we can't possibly try every single combination of nudging the coefficients (5^25 = 3x10^17, three hundred quadrillion or three hundred million billion possible combinations. The universe is around 4.4 hundred quadrillion seconds old, the Sun is around 1.5 hundred quadrillion seconds old) So try two combinations a second since the Sun formed and we'd only just have finished. 

But can manage 5 coeffs at a time (5^5 = 3125 combinations).

For each deprivation quantile separately, find the nudged coefficients that give the best fit in that quantile. But see that combining five good fits gives very poor fits to the total numbers of admissions from each age band.

Try the same for the age bands, but a similar result: can fit well to age but the combined good fits are bad fits to each deprivation quantile.

So have to consider all 25 coefficients together. No way to simplify the problem to 5 coeffs only.
Also expect that most of these three hundred quadrillion combinations would be rubbish. So find a way to try considerably fewer combinations, pick out only those that are likely to be any good.

Assume that the starting values are pretty good fits to the data and so that increasing or decreasing the value of only one coefficient is bound to make the fitness worse. Instead, force two changes at once: one coefficient must increase and another must decrease, and the two must be either in the same row (same deprivation level) or in the same column (same age band). Then pick the pair of changes that results in the best overall fitness. Continue to change either a pair or just one coefficient at once until no further improvements can be made. So in each pass, try a number of combinations, but not all three hundred quadrillion options, just the ones that are quite close to the starting values.

Try this walking nudge method on each of the 100 outputs from the genetic algorithm, and so find the best set of coefficients that is near to those starting points.

Results: 7 of the 100 sets converge on one set of values that have better fitness than the other 93 sets manage. Go with these as the "best" values. 

In [1]:
import polars as pl
import os
import numpy as np
import itertools

## Load data

### Admissions

In [2]:
path_to_msoa_stats = os.path.join('data', 'msoa_cleaned.csv')

df_stats = pl.read_csv(path_to_msoa_stats)

In [3]:
df_stats.head()

MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion,total_health,age_less65,age_65,age_70,age_75,age_over80,depriv_quantile_min,depriv_quantile_max
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64
"""Adur 001""",14.333333,16.924833,8815,"""E""",6799,1251,474,0.79763,0.146762,0.055608,"""E02006534""",0.0559,0.0528,0.0422,0.7872,0.062,8524,6939.168,492.7585,465.432,371.993,546.53,0.4,0.6
"""Adur 002""",7.333333,6.4704,7263,"""E""",5537,838,259,0.83464,0.126319,0.039041,"""E02006535""",0.0578,0.0774,0.0492,0.7467,0.0692,6634,5423.2821,419.8014,562.1562,357.3396,502.5996,0.8,1.0
"""Adur 003""",9.333333,13.7334,7354,"""E""",5820,969,311,0.819718,0.136479,0.043803,"""E02006536""",0.0609,0.0582,0.0421,0.7729,0.0661,7100,5683.9066,447.8586,428.0028,309.6034,486.0994,0.6,0.8
"""Adur 004""",21.0,26.199857,10582,"""E""",7872,1546,709,0.777328,0.152661,0.070011,"""E02006537""",0.0465,0.0438,0.0367,0.8091,0.0638,10127,8561.8962,492.063,463.4916,388.3594,675.1316,0.2,0.4
"""Adur 005""",13.666667,11.7948,9059,"""E""",7106,1081,339,0.833451,0.126789,0.039761,"""E02006538""",0.0597,0.067,0.0425,0.7643,0.0662,8526,6923.7937,540.8223,606.953,385.0075,599.7058,0.6,0.8


Pick out column names for the health and age proportions:

In [70]:
props_age = [
    'age_less65_proportion', 'age_65_proportion', 'age_70_proportion',
    'age_75_proportion', 'age_over80_proportion'
]
age_numbers = [p.replace('_proportion', '') for p in props_age]
qmin_list = sorted(df_stats['depriv_quantile_min'].unique())

# Names of coeffs:
coeff_names = [f'{a}_q{str(round(q, 1)).replace(".", "")}'
               for q in qmin_list for a in age_numbers]
# Check the first few:
print(coeff_names[:3])

# Quantile names:
quantile_str_list = sorted(list(set([c.split('_')[-1] for c in coeff_names])))

['age_less65_q00', 'age_65_q00', 'age_70_q00']


Gather admissions data by deprivation quantile:

In [5]:
admissions_lists = []
x_lists = []

for qmin in qmin_list:
    df_stats_here = df_stats.filter(df_stats['depriv_quantile_min'] == qmin)
    # MSOA data in the same order as those coefficients:
    x_lists_here = [df_stats_here[a] for a in age_numbers]
    admissions_here = df_stats_here['admissions'].to_numpy().tolist()
    # Store:
    admissions_lists.append(admissions_here)
    x_lists.append(x_lists_here)

### Age-admissions coefficients

Starting SSNAP coefficients:

In [6]:
df_pop_admissions = pl.read_csv(os.path.join('outputs', 'ssnap_coeffs.csv'))

In [7]:
df_pop_admissions

Age Groups,population,prop_of_all_pop,count,prop_of_all_admissions,admissions_annual,admissions_annual_boost,prob_stroke_given_age
str,i64,f64,i64,f64,f64,f64,f64
"""Under 65""",45933245,0.816055,38827,0.231449,12942.33333,18737.66819,0.000408
"""65-69""",2796740,0.049687,15324,0.091347,5108.0,7395.26689,0.002644
"""70-74""",2779326,0.049378,21508,0.12821,7169.33333,10379.62674,0.003735
"""75-79""",1940686,0.034478,24150,0.143959,8050.0,11654.63948,0.006005
"""80 and over""",2836964,0.050402,67947,0.405035,22649.0,32790.7987,0.011558


Pick out SSNAP coefficients:

In [8]:
coeffs_ssnap = df_pop_admissions['prob_stroke_given_age'].to_numpy()

coeffs_ssnap

array([0.000408, 0.002644, 0.003735, 0.006005, 0.011558])

Turn them into a dictionary:

In [9]:
labels = ['less65', '65', '70', '75', 'over80']
coeffs_ssnap_dict = (
    dict(zip(labels, df_pop_admissions['prob_stroke_given_age'].to_numpy())))

coeffs_ssnap_dict

{'less65': 0.000408,
 '65': 0.002644,
 '70': 0.003735,
 '75': 0.006005,
 'over80': 0.011558}

Pick out admissions numbers:

In [10]:
admissions_by_age = (
    df_pop_admissions['admissions_annual_boost'].to_numpy().flatten())

### Best results from genetic algorithm

Data stored as scale factors for the SSNAP-derived coefficients.

In [11]:
df_best_gens = pl.read_csv(os.path.join('outputs', 'best_inds_deap.csv'))

In [12]:
df_best_gens.head()

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,fitness,r2_all,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed00""",33.0,1.3,1.2,1.332,1.237,1.2,1.026,1.145,1.151,1.041,1.1,1.0,1.0,0.988,0.938,1.0,0.862,0.9,0.945,0.938,0.937,0.8,0.9,0.732,0.832,0.874,238.372,0.584365,0.502719,0.579147,0.616005,0.598245,0.606993
"""randomseed01""",33.0,1.1,1.3,1.305,1.3,1.231,1.1,1.1,1.1,1.023,1.1,1.0,1.0,1.0,1.0,1.0,0.969,0.9,0.9,0.892,0.9,0.837,0.849,0.8,0.8,0.894,238.507,0.575882,0.471584,0.579147,0.616005,0.583824,0.606993
"""randomseed02""",43.0,1.273,1.308,1.175,1.2,1.214,0.981,1.1,1.103,1.1,1.131,0.949,0.987,1.046,1.0,1.0,0.949,0.979,0.896,0.9,0.9,0.844,0.773,0.861,0.8,0.88,238.037,0.58383,0.485431,0.583466,0.616005,0.606683,0.606993
"""randomseed03""",28.0,1.171,1.3,1.327,1.186,1.3,1.1,1.198,1.1,1.1,1.1,1.042,1.0,1.0,1.0,0.971,0.881,0.9,0.894,1.0,0.9,0.8,0.771,0.808,0.7,0.9,239.49,0.585993,0.508469,0.583466,0.631482,0.605116,0.580367
"""randomseed04""",39.0,1.2,1.2,1.25,1.225,1.293,1.1,1.1,1.11,1.069,1.1,0.976,1.019,1.0,1.0,0.961,0.925,0.904,0.9,0.915,0.91,0.8,0.9,0.844,0.8,0.879,238.453,0.591769,0.508469,0.579147,0.631482,0.612783,0.606993


Convert the scale factors to the actual age-deprivation coefficient values using the SSNAP coefficients.

In [13]:
for coeff in coeff_names:
    key = coeff.split('_')[1]
    new_data = df_best_gens[coeff] * coeffs_ssnap_dict[key]
    
    df_best_gens = df_best_gens.with_columns(pl.Series(coeff, new_data))

Round results:

In [14]:
# Set up dictionary with how many decimal places to round to:
labels = ['less65', '65', '70', '75', 'over80']
round_dict = dict(zip(labels, [4, 3, 3, 3, 3]))

for coeff in coeff_names:
    key = coeff.split('_')[1]
    new_data = np.round(df_best_gens[coeff], round_dict[key])
    
    df_best_gens = df_best_gens.with_columns(pl.Series(coeff, new_data))

Drop the measures of goodness of fit now that the coefficients have been rounded:

In [15]:
df_best_gens = df_best_gens.drop(
    ['fitness'] + [c for c in df_best_gens.columns if c.startswith('r2')])

## Recalculate fitness

In [16]:
def predict_admissions(x_lists, coeffs):
    """
    x_lists: np.array.
    coeffs: np.array.
    
    Have to have same number of coeffs as x_lists.
    """
    # Predicted admissions:
    # yhat = (x_lists * coeffs.reshape(len(coeffs), 1)).sum(axis=0)
    yhat = sum(
        [x_lists[i] * coeffs[i] for i in range(len(coeffs))]
    )
    return yhat.to_numpy().tolist()

In [17]:
def predict_admissions_each_age(x_lists, coeffs):
    """
    x_lists: np.array.
    coeffs: np.array.
    
    Have to have same number of coeffs as x_lists.
    """
    # Predicted admissions:
    yhat = (
        [(x_lists[i] * coeffs[i]).sum() for i in range(len(coeffs))]
    )
    return yhat

Goodness check option 1: This calculates the sum of the square of the differences between predicted and actual admission numbers:

In [18]:
def find_square_residuals(yhat, y):
    # Difference from actual:
    sqres = (np.array(yhat) - np.array(y))**2.0
    # Sum of differences:
    sum_sqres = sqres.sum()
    return np.sqrt(sum_sqres)

In [101]:
def many_sum_sqres(individual, x_lists, admissions_lists):
    # Predictions for each MSOA:
    predictions_lists = []
    sum_sqres_by_depriv = []
    for i in range(5):
        coeffs = individual[(i*5):(i*5)+5]  # coeffs for this depriv.
        yhat = predict_admissions(x_lists[i], coeffs)
        sum_sqres = find_square_residuals(yhat, admissions_lists[i])
        predictions_lists.append(yhat)
        sum_sqres_by_depriv.append(sum_sqres)


    # Combine lists:
    observed_all = sum(admissions_lists, [])
    predicted_all = sum(predictions_lists, [])
    # Calculate sum of square residuals across all MSOA:
    sum_sqres = find_square_residuals(predicted_all, observed_all)

    return sum_sqres, sum_sqres_by_depriv

In [102]:
# the goal ('fitness') function to be maximized
def eval_admissions(individual, admissions_lists, x_lists, admissions_by_age):

    # Predictions for each age band across England:
    predictions_by_age = [0.0] * 5
    for i in range(5):
        # Population numbers for areas in this quantile:
        x_lists_here = x_lists[i]
        # Separate prediction for each deprivation quantile:
        coeffs = individual[(i*5):(i*5)+5] #* np.array(coeffs_ssnap)
        yhat_list = predict_admissions_each_age(x_lists_here, coeffs)
        # predictions_lists.append(yhat)
        for j, y in enumerate(yhat_list):
            predictions_by_age[j] += y
    # Divide the admissions by age band by the observed values:
    for j in range(len(predictions_by_age)):
        predictions_by_age[j] /= admissions_by_age[j]
    # Wrongness ratio factor:
    rat = sum(np.abs(np.array(predictions_by_age) - 1.0)) + 1.0

    sum_sqres, sum_sqres_by_depriv = many_sum_sqres(individual, x_lists, admissions_lists)
    
    return (sum_sqres, rat, sum_sqres * rat, predictions_by_age, sum_sqres_by_depriv)

Recalculate fitnesses for the genetic algorithm outputs:

In [20]:
def eval_fitness_to_df(df_best_gens, admissions_lists, x_lists, admissions_by_age, coeff_names, concat=False):
    eval_dict = {}
    eval_labels = [
        'sum_sqres',
        'wrong_rat',
        'fitness',
        'wrong_rat_under65',
        'wrong_rat_65',
        'wrong_rat_70',
        'wrong_rat_75',
        'wrong_rat_over80',
        'sum_sqres_q00',
        'sum_sqres_q02',
        'sum_sqres_q04',
        'sum_sqres_q06',
        'sum_sqres_q08',
    ]
    eval_dict = dict(zip(eval_labels, [[] for e in eval_labels]))
    
    for d in range(len(df_best_gens)):
        df = df_best_gens[d]
        # Pick out coefficients:
        coeffs = df[coeff_names].to_numpy().flatten()
        sum_sqres, rat, fitness, ratios_by_age, sum_sqres_by_depriv = eval_admissions(
            coeffs,
            admissions_lists,
            x_lists,
            admissions_by_age
        )
        eval_dict['sum_sqres'].append(sum_sqres)
        eval_dict['wrong_rat'].append(rat)
        eval_dict['fitness'].append(fitness)
        eval_dict['wrong_rat_under65'].append(ratios_by_age[0])
        eval_dict['wrong_rat_65'].append(ratios_by_age[1])
        eval_dict['wrong_rat_70'].append(ratios_by_age[2])
        eval_dict['wrong_rat_75'].append(ratios_by_age[3])
        eval_dict['wrong_rat_over80'].append(ratios_by_age[4])
        eval_dict['sum_sqres_q00'].append(sum_sqres_by_depriv[0])
        eval_dict['sum_sqres_q02'].append(sum_sqres_by_depriv[1])
        eval_dict['sum_sqres_q04'].append(sum_sqres_by_depriv[2])
        eval_dict['sum_sqres_q06'].append(sum_sqres_by_depriv[3])
        eval_dict['sum_sqres_q08'].append(sum_sqres_by_depriv[4])

    # Either start a new blank dataframe or add the fitnesses to the
    # input df:
    df_new = df_best_gens if concat else pl.DataFrame()
    for key, v in eval_dict.items():
        df_new = df_new.with_columns(pl.Series(key, v))
    return df_new

In [21]:
df_best_gens = eval_fitness_to_df(df_best_gens, admissions_lists, x_lists, admissions_by_age, coeff_names, concat=True)

View results:

In [22]:
df_best_gens.sort('fitness').head()

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed74""",22.0,0.0005,0.004,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.191107,1.055834,252.546109,0.980725,1.01735,1.013293,0.99959,0.994494,111.087199,107.849722,108.96863,104.473782,102.232696
"""randomseed10""",26.0,0.0005,0.004,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,240.357618,1.052517,252.980542,0.980725,1.01735,0.990024,0.99959,0.994494,110.501962,107.849722,108.96863,107.721177,102.232696
"""randomseed94""",18.0,0.0005,0.004,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,238.67206,1.061548,253.361768,0.980725,1.01735,0.990024,0.99959,1.014536,110.501962,107.849722,108.96863,103.905822,102.232696
"""randomseed26""",20.0,0.0005,0.004,0.005,0.008,0.014,0.0004,0.003,0.004,0.006,0.014,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.532221,1.061076,254.161938,0.980725,1.01735,0.990024,0.994136,0.991389,110.922311,107.800573,106.750111,107.721177,102.232696
"""randomseed57""",56.0,0.0005,0.004,0.005,0.007,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.006,0.01,0.0003,0.002,0.003,0.005,0.01,239.371525,1.062211,254.262948,0.980725,1.01735,0.990024,1.010103,0.994494,110.501962,108.40745,108.96863,104.929413,102.232696


Pick out the best coeffs so far:

In [78]:
df_coeffs_best_deap = df_best_gens.sort('fitness')[0]
coeffs_best_deap = df_coeffs_best_deap[coeff_names].to_numpy().flatten()

## Tests

### Test 1: manual tweaking of coefficients

can other combos be better?

From the table above we can see that the best overall combo is not made up of the best combo for each deprivation quantile:

In [30]:
sum_sqres_cols = [c for c in df_best_gens.columns if c.startswith('sum_sqres')]

d1 = df_best_gens.sort('sum_sqres')[0][sum_sqres_cols]
d2 = df_best_gens[sum_sqres_cols].min()

d1 = d1.with_columns(pl.Series('coeff_combo', ['max_sum_sqres_overall']))
d2 = d2.with_columns(pl.Series('coeff_combo', [f'max_sum_sqres_each_depriv']))

display(pl.concat((d1, d2)))

sum_sqres,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08,coeff_combo
f64,f64,f64,f64,f64,f64,str
237.454911,110.501962,107.849722,106.276266,103.905822,102.232696,"""max_sum_sqres_overall"""
237.454911,110.501962,107.800573,106.276266,103.472366,99.88842,"""max_sum_sqres_each_depriv"""


The second row of the table has the best r-squared for each deprivation quantile. Its values are higher for the second and the two most-deprived quantiles than in the overall best combo.

View the best combo for these deprivation quantiles:

In [31]:
for q in ['q02', 'q06', 'q08']:
    coeffs_q = [c for c in df_best_gens.columns if q in c]
    mask = df_best_gens.sort('sum_sqres')[f'sum_sqres_{q}'] == df_best_gens[f'sum_sqres_{q}'].min()

    d1 = df_best_gens.sort('sum_sqres')[0][coeffs_q]
    d2 = df_best_gens.sort('sum_sqres').filter(mask)[0][coeffs_q]
    d1 = d1.with_columns(pl.Series('coeff_combo', ['best_overall']))
    d2 = d2.with_columns(pl.Series('coeff_combo', [f'best_{q}']))

    display(pl.concat((d1, d2)))

age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,sum_sqres_q02,coeff_combo
f64,f64,f64,f64,f64,f64,str
0.0004,0.003,0.004,0.007,0.013,107.849722,"""best_overall"""
0.0004,0.003,0.004,0.006,0.014,107.800573,"""best_q02"""


age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,sum_sqres_q06,coeff_combo
f64,f64,f64,f64,f64,f64,str
0.0004,0.002,0.003,0.005,0.011,103.905822,"""best_overall"""
0.0004,0.002,0.003,0.006,0.011,103.472366,"""best_q06"""


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres_q08,coeff_combo
f64,f64,f64,f64,f64,f64,str
0.0003,0.002,0.003,0.005,0.01,102.232696,"""best_overall"""
0.0003,0.002,0.003,0.005,0.011,99.88842,"""best_q08"""


Manually adjust the values in the best combo to recreate the best r-squared for each deprivation quantile.

In [32]:
# Pick out coeffs:
coeffs = df_best_gens.filter(df_best_gens['sum_sqres'] == df_best_gens['sum_sqres'].min())[coeff_names].to_numpy().flatten()
# Update some:
coeffs[coeff_names.index('age_75_q02')] = 0.006
coeffs[coeff_names.index('age_over80_q02')] = 0.014
coeffs[coeff_names.index('age_75_q06')] = 0.006
coeffs[coeff_names.index('age_over80_q08')] = 0.011

Check that coeffs still increase with age (across) and deprivation level (upwards):

In [33]:
coeffs.reshape(5, 5)

array([[0.0005, 0.004 , 0.005 , 0.007 , 0.014 ],
       [0.0004, 0.003 , 0.004 , 0.006 , 0.014 ],
       [0.0004, 0.002 , 0.004 , 0.006 , 0.012 ],
       [0.0004, 0.002 , 0.003 , 0.006 , 0.011 ],
       [0.0003, 0.002 , 0.003 , 0.005 , 0.011 ]])

What's the effect on overall R^2?

In [35]:
sum_sqres, rat, fitness, ratios_by_age, sum_sqres_by_depriv = eval_admissions(
    coeffs,
    admissions_lists,
    x_lists,
    admissions_by_age
)

# Best R^2 from all genetic algorithm results:
fitness_best = df_best_gens['fitness'].min()
sum_sqres_best = df_best_gens['sum_sqres'].min()

print(f'New sum_sqres: {sum_sqres:.4f}')
print(f'Old sum_sqres: {sum_sqres_best:.4f}')

print(f'New fitness: {fitness:.4f}')
print(f'Old fitness: {fitness_best:.4f}')

New sum_sqres: 236.2425
Old sum_sqres: 237.4549
New fitness: 271.9798
Old fitness: 252.5461


The overall sum of square residuals has decreased with this manual changing of the coefficients - good - but the overall fitness taking into account numbers of admissions in each age band has increased - bad.

It is worth trying more combinations of coefficients near the found values to check whether any small adjustments can make better results. The genetic algorithm isn't likely to have tried every good combination by chance, and there could be other better options yet to be discovered.

### Test 2: Random offsets

Are there any combos of coefficients where the fitness is better than the current best?

Can't try every combination, so here pick some out at random and see if any are better by chance.

Allow these offsets for the coeffs:

In [36]:
offsets = np.array([1e-4, 1e-3, 1e-3, 1e-3, 1e-3] * 5)

What probability of no change do we need to have an average of 3 coeffs changing?

In [37]:
((25.0 - 3.0)/25.0)

0.88

Generate 10,000 lists of changes to the starting parameters. Each change may be either zero, or plus or minus one or two steps from the start value. Each coefficient has an 88% chance of not changing, a 4% chance of moving up or down one step, and a 2% chance of moving up or down two steps.

In [38]:
seeds_list = []
random_offsets = []
for seed in np.arange(10000):
    np.random.seed(seed)
    r = list(offsets * np.random.choice(np.arange(-2, 3, 1), 25, p=[0.02, 0.04, 0.88, 0.04, 0.02]))
    if r not in random_offsets:
        random_offsets.append(r)
        seeds_list.append(seed)

Check how many sets of changes were kept after removing repeats:

In [39]:
len(random_offsets)

7730

For each set in turn, apply the changes to the set of starting parameters (here the best one of the 100 sets of outputs from the genetic algorithm). Check that the resulting coefficients obey the rules: they must increase with age band and with deprivation level. If this check is passed, then keep a copy of the resulting coefficients.

In [40]:
coeffs = coeffs_best_deap
results = [list(coeffs)]
seeds_used = [-42]

for s, seed in enumerate(seeds_list):
    r = coeffs + random_offsets[s]
    # Check if rules are met - coeffs increase with age and depriv:
    r_arr = r.reshape(5, 5)
    age_increases = (np.diff(r_arr, axis=1) >= 0).all()
    depriv_increases = (np.diff(r_arr, axis=0) <= 0).all()

    if age_increases & depriv_increases:
        if list(r) not in results:
            results.append(list(r))
            seeds_used.append(seed)

# Turn the results into a DataFrame:
df_nudged = pl.DataFrame(results, schema=coeff_names, orient='row')
df_nudged = df_nudged.insert_column(0, pl.Series('seed', seeds_used))

Check how many valid sets of coefficients there are:

In [41]:
len(df_nudged)

640

View the first few sets of coefficients:

In [42]:
df_nudged.head()

seed,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-42,0.0005,0.004,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01
2,0.0005,0.003,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01
27,0.0005,0.004,0.004,0.007,0.014,0.0005,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01
32,0.0005,0.004,0.004,0.008,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01
37,0.0006,0.004,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.007,0.012,0.0004,0.002,0.004,0.005,0.01,0.0002,0.002,0.003,0.005,0.01


Evaluate the fitnesses:

In [43]:
df_nudged = eval_fitness_to_df(df_nudged, admissions_lists, x_lists, admissions_by_age, coeff_names, concat=True)

Only keep the sets of coefficents where the fitness is at least as good as the starting value:

In [45]:
fitness_orig = df_nudged.filter(df_nudged['seed'] == -42)['fitness'].to_numpy()[0]
mask = df_nudged['fitness'] <= fitness_orig

View these better coefficients' fitness values:

In [46]:
df_nudged.filter(mask).sort('fitness')[['seed', 'sum_sqres', 'wrong_rat', 'fitness']]

seed,sum_sqres,wrong_rat,fitness
i64,f64,f64,f64
6483,238.324924,1.045778,249.234929
9196,239.490857,1.050797,251.656359
540,241.387779,1.043606,251.913656
3575,237.8619,1.060951,252.359804
4131,239.769863,1.052534,252.365853
-42,239.191107,1.055834,252.546109


The best set of coefficients has both a better sum of square residuals and a better age-band wrongness ratio than the starting set.

View the better sets of coefficients as a grid:

In [51]:
# Pick out original coefficients before any nudging:
coeffs_orig = df_nudged.filter(df_nudged['seed'] == -42)[coeff_names].to_numpy().flatten().reshape(5, 5)

df = df_nudged.filter(mask).sort('fitness')
for i in range(len(df)):
    # Pick out fitness score and coefficients here:
    fitness_here = df_nudged.sort('fitness')[i]['fitness'].to_numpy().flatten()[0]
    coeffs_here = df_nudged.sort('fitness')[i][coeff_names].to_numpy().flatten().reshape(5, 5)
    # Display results:
    print(f'Fitness: {fitness_here:.2f}')
    print(coeffs_here)
    print('Change from starting coefficients:')
    print(coeffs_here - coeffs_orig)
    print('\n')

Fitness: 249.23
[[0.0005 0.004  0.004  0.007  0.014 ]
 [0.0004 0.004  0.004  0.007  0.013 ]
 [0.0004 0.002  0.004  0.006  0.012 ]
 [0.0004 0.002  0.004  0.005  0.01  ]
 [0.0003 0.002  0.003  0.005  0.01  ]]
Change from starting coefficients:
[[ 0.     0.     0.     0.     0.   ]
 [ 0.     0.001  0.     0.     0.   ]
 [ 0.    -0.001  0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]]


Fitness: 251.66
[[0.0005 0.004  0.004  0.007  0.014 ]
 [0.0004 0.003  0.004  0.007  0.012 ]
 [0.0004 0.003  0.004  0.006  0.012 ]
 [0.0004 0.002  0.004  0.005  0.011 ]
 [0.0003 0.002  0.003  0.005  0.01  ]]
Change from starting coefficients:
[[ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.    -0.001]
 [ 0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.001]
 [ 0.     0.     0.     0.     0.   ]]


Fitness: 251.91
[[0.0005 0.004  0.004  0.007  0.014 ]
 [0.0004 0.004  0.004  0.007  0.013 ]
 [0.0004 0.003  0.004  0.006  0.012 ]


Now in principle there are combinations near the starting values that have better fitness. Check all sensible combinations of nudged coefficeints to winkle out the best one.

Note that these good combinations mostly follow a pattern: there is a pair of changed coefficients that share a row or a column, and one of the pair has increased slightly and the other decreased slightly.

### Test 3: Optimise deprivation quantiles

can we pick out a few combos that are good for each depriv quantile, then put five quantiles together to get 25 coeffs that have good overall fitness?

Grid near best combo

Take the best coefficients. For each deprivation quantile, nudge the best coefficient one or two clicks either way and calculate the new fitness scores.

In [65]:
start_coeffs = coeffs_best_deap

start_coeffs.reshape(5, 5)

array([[0.0005, 0.004 , 0.004 , 0.007 , 0.014 ],
       [0.0004, 0.003 , 0.004 , 0.007 , 0.013 ],
       [0.0004, 0.003 , 0.004 , 0.006 , 0.012 ],
       [0.0004, 0.002 , 0.004 , 0.005 , 0.01  ],
       [0.0003, 0.002 , 0.003 , 0.005 , 0.01  ]])

Make every combination of these start coefficients nudged up or down one or two steps:

In [71]:
def generate_new_combos_depriv(coeffs_this_depriv):
    offsets = np.array([1e-4, 1e-3, 1e-3, 1e-3, 1e-3])
    
    # Add on the offsets to make the new options:
    # Set up each value to be nudged one or two sig figs up or down.
    new_options = [
        [round(coeffs_this_depriv[j] + i*offsets[j], 4) for i in range(-2, 3)] for j in range(5)
    ]
    # Generate all combinations of these new parameters.
    # Each element of this list is a tuple of five new coeffs.
    all_new_option_combos = list(itertools.product(*new_options))
    # Only keep combinations where coefficients increase with age band:
    mask = (np.diff(all_new_option_combos, axis=1) >= 0).all(axis=1)
    all_new_option_combos = np.array(all_new_option_combos)[mask]
    return all_new_option_combos

Find every nudged variation on the best coefficients and calculate the new r-squared for that deprivation quantile.

In [72]:
new_combo_df_dict = {}

for quantile_str in quantile_str_list:
    # Set up final column names:
    depriv_ind = quantile_str_list.index(quantile_str)
    best_coeff_cols = [c for c in coeff_names if quantile_str in c]
    col_sqres = f'sum_sqres_{quantile_str}'

    # Find all nudged coefficients for this deprivation band:
    coeffs_this_depriv = start_coeffs[depriv_ind*5:depriv_ind*5+5]
    all_new_option_combos = generate_new_combos_depriv(coeffs_this_depriv)

    # Calculate fitnesses of each of the nudged sets.
    # Store results in here:
    list_sum_sqres = []
    for c, coeffs in enumerate(all_new_option_combos):
        # Population numbers for areas in this quantile:
        # Separate prediction for each deprivation quantile:
        yhat = predict_admissions(x_lists[depriv_ind], np.array(list(coeffs)))
        # Compare with the observed admissions to calculate sum of square residuals:
        sum_sqres = find_square_residuals(yhat, admissions_lists[depriv_ind])
        # Store result:
        list_sum_sqres.append(sum_sqres)

    # Place new coeff combos into dataframe:
    df_new_combos = pl.DataFrame(all_new_option_combos, schema=best_coeff_cols, orient='row')
    # Make a new column with the sum of square residuals values:
    df_new_combos = df_new_combos.with_columns(pl.Series(col_sqres, list_sum_sqres))
    
    new_combo_df_dict[quantile_str] = df_new_combos

View the results:

In [90]:
for quantile_str in list(new_combo_df_dict.keys()):
    print(quantile_str)
    # Print the best coefficients for comparison:
    cols = [c for c in coeff_names if quantile_str in c] + [f'sum_sqres_{quantile_str}', 'sum_sqres']
    print('Coeffs in best overall combo:')
    display(df_coeffs_best_deap[cols])
    # Print the nudged coeffs from best to worst:
    print('Nudged coeffs:')
    print(f'(best options out of {len(new_combo_df_dict[quantile_str])} total)')
    display(new_combo_df_dict[quantile_str].sort(f'sum_sqres_{quantile_str}').head())
    print('\n'*2)

q00
Coeffs in best overall combo:


age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,sum_sqres_q00,sum_sqres
f64,f64,f64,f64,f64,f64,f64
0.0005,0.004,0.004,0.007,0.014,111.087199,239.191107


Nudged coeffs:
(best options out of 1750 total)


age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,sum_sqres_q00
f64,f64,f64,f64,f64,f64
0.0004,0.006,0.006,0.006,0.014,110.354891
0.0005,0.005,0.005,0.005,0.014,110.392239
0.0004,0.006,0.006,0.007,0.013,110.402224
0.0004,0.005,0.006,0.006,0.015,110.403983
0.0004,0.005,0.005,0.006,0.016,110.429618





q02
Coeffs in best overall combo:


...

age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,sum_sqres_q06
f64,f64,f64,f64,f64,f64
0.0003,0.002,0.003,0.006,0.012,102.55868
0.0003,0.001,0.003,0.007,0.012,102.588027
0.0003,0.001,0.004,0.006,0.012,102.588145
0.0003,0.002,0.002,0.007,0.012,102.623322
0.0003,0.003,0.003,0.005,0.012,102.737814





q08
Coeffs in best overall combo:


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres_q08,sum_sqres
f64,f64,f64,f64,f64,f64,f64
0.0003,0.002,0.003,0.005,0.01,102.232696,239.191107


Nudged coeffs:
(best options out of 1450 total)


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres_q08
f64,f64,f64,f64,f64,f64
0.0001,0.003,0.004,0.004,0.012,98.631977
0.0002,0.001,0.004,0.005,0.012,98.692418
0.0002,0.002,0.003,0.005,0.012,98.702951
0.0001,0.002,0.003,0.007,0.012,98.709033
0.0001,0.003,0.003,0.006,0.012,98.709283


For all deprivation quantiles, the sets of nudged coeffs contain an option with lower sum of square residuals than the one in the best overall combo.

Gather the coefficients that are best for each deprviation quantile:

In [81]:
best_combos = []
best_combo_cols = []

for q, quantile_str in enumerate(list(new_combo_df_dict.keys())):
    df = new_combo_df_dict[quantile_str].sort(f'sum_sqres_{quantile_str}')
    df = df[0] if q < 4 else df[3]
    cols = [c for c in df.columns if c.startswith('age')]
    best_combo_cols += cols
    best_combos += list(df[cols].to_numpy().flatten())

Check that the complete set of coefficients meet the requirements of increasing with age and deprivation:

In [88]:
print(np.array(best_combos).reshape(5, 5))

print('\nCheck coeffs increase with age band (across):')
print(np.diff(np.array(best_combos).reshape(5, 5), axis=1) >= 0)

print('\nCheck coeffs increase with deprivation (downwards):')
print(np.diff(np.array(best_combos).reshape(5, 5), axis=0) <= 0)

[[0.0004 0.006  0.006  0.006  0.014 ]
 [0.0004 0.003  0.004  0.006  0.014 ]
 [0.0003 0.002  0.002  0.007  0.014 ]
 [0.0003 0.002  0.003  0.006  0.012 ]
 [0.0001 0.002  0.003  0.007  0.012 ]]

Check coeffs increase with age band (across):
[[ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]]

Check coeffs increase with deprivation (downwards):
[[ True  True  True  True  True]
 [ True  True  True False  True]
 [ True  True False  True  True]
 [ True  True  True False  True]]


These requirements are not met! The values break the pattern in a handful of places.

Instead pick more carefully to ensure these conditions are met.

It doesn't work!

Instead, try a few combinations of "best" results. For each deprivation quantile in turn, use its best results and then select results for the other quantiles that work around it.

The following functions search an array of coefficients to find sets that work with a given set of coefficients. They find either coefficients that always increase or stay the same as the reference set, or that always decrease or stay the same.

In [91]:
def pick_conditions_met(combo, all_combos, pick_more=True):
    """
    Find options where all coeffs are more than/same as fixed values.

    Inputs
    ------
    combo      - list. Fixed coefficients.
    all_combos - np.array. One row per set of coefficients.
                 Assume sorted from best to worst r-squared.
    Returns
    -------
    list. The selected coefficients from the big list.
    """
    # List of masks, one mask per age band.
    # Masks are for whether the coeffs for this age band
    # are more or equal to the fixed value for this age band.
    if pick_more:
        masks = [(all_combos[:, p] >= combo[p]) for p in range(len(combo))]
    else:
        masks = [(all_combos[:, p] <= combo[p]) for p in range(len(combo))]
    # Gather masks:
    masks = np.vstack(masks).T
    # Check where condition is met for all age bands:
    valid_mask = masks.all(axis=1)
    # Select this index "pick", the first in the list where the
    # condition is met for all age bands.
    pick = np.where(valid_mask == True)[0][0]
    # Return a list of the coefficients in this row of the table:
    return all_combos[pick]

Pick out combos:

In [98]:
def pick_combos(fixed_combo, new_combo_100_df_dict, q):
    """
    Pick combos of all good coeffs that meet rules.

    Fix the coefficients for the qth quantile in the list.
    Then pick coefficients for the adjacent quantiles that meet rules
    and continue until all quantiles have been picked.
    Picked coefficients must increase with age band and with level
    of deprivation.

    Inputs
    ------
    fixed_combo           - list. Five coefficients that are fixed.
    new_combo_100_df_dict - dict. One entry per deprivation quantile.
                            Each entry is a dataframe of coefficients
                            and their r-squared value.
    q                     - int. Index of depriv quantile for the
                            fixed combo.

    Returns
    -------
    best_combo - list. Length 25, picked coefficients.
    """
    def gather_coeff_combos(p):
        """
        Change big dataframe of coeff combos to array for picking.
        """
        # Find the name of the r-squared column for the next combo:
        p_str = list(new_combo_100_df_dict.keys())[p]
        # Pick out the coefficients and r-squared for this combo:
        arr = new_combo_100_df_dict[p_str].sort(f'sum_sqres_{p_str}').to_numpy()
        # Cut off r-squared column:
        arr = arr[:, :-1]
        return arr
        
    # Set up coeff storage.
    # Store the coeffs for each quantile in their own list in order
    # in the following list of lists. Start with empty lists...
    best_combo = [[]] * 5
    # ... fill in one list of coeffs...
    best_combo[q] = list(fixed_combo)
    # ... then use that filled list to select the adjacent lists.
    for p in range(q+1, 5):
        # Find the combos where values should be less than or same as the
        # starting values.
        arr = gather_coeff_combos(p)
        # Find the best coeff combo that meets the decreasing criteria:
        picked_coeffs = pick_conditions_met(best_combo[p-1], arr, pick_more=False)
        # Store result:
        best_combo[p] = list(picked_coeffs)
    for p in range(q-1, -1, -1):
        # Find the combos where values should be more than or same as the
        # starting values.
        arr = gather_coeff_combos(p)
        # Find the best coeff combo that meets the increasing criteria:
        picked_coeffs = pick_conditions_met(best_combo[p+1], arr)
        # Store result:
        best_combo[p] = list(picked_coeffs)
    # Return a single flat list of coeffs, length 25.
    best_combo = sum(best_combo, [])
    return best_combo

Run this function for each quantile in turn. For each loop, a different quantile's coefficients are fixed first. Then the other quantiles' coefficients have to work around those fixed values.

In [99]:
best_picked_combos = {}
# Each item will be a list of 25 coeffs.

for q, quantile_str in enumerate(list(new_combo_df_dict.keys())):
    # Pick out a list of five coefficients for this depriv quantile:
    fixed_combo = (new_combo_df_dict[quantile_str]
                   .sort(f'sum_sqres_{quantile_str}')[0]
                   .to_numpy().flatten()[:-1])
    # Pick out coeffs for the other quantiles that work well with this:
    best_picked_combos[quantile_str] = pick_combos(
        fixed_combo, new_combo_df_dict, q)

Calculate their sum of square residuals:

In [103]:
best_picked_combos_r2s = {}

for quantile_str, best_combo in best_picked_combos.items():
    # Calculate fitnesses of each of the nudged sets.
    sum_sqres, sum_sqres_by_depriv = many_sum_sqres(best_combo, x_lists, admissions_lists)
    best_picked_combos_r2s[quantile_str] = sum_sqres_by_depriv + [sum_sqres]

Rearrange the picked coefficients into dataframes:

In [104]:
best_picked_combos_dfs = {}
age_label_list = ['less65', '65-70', '70-75', '75-80', 'over80']

for quantile_str, best_combo in best_picked_combos.items():
    # Make a dataframe:
    data = np.array(best_combo).reshape(5, 5)
    # Add deprivation column:
    data = np.hstack((np.array(quantile_str_list).reshape(5, 1), data))
    df = pl.DataFrame(data, schema=['depriv_quantile_min'] + age_label_list)
    for col in age_label_list:
        df = df.with_columns(pl.col(col).cast(float))
    # Place r-squared results in the dataframe:
    list_sum_sqres = best_picked_combos_r2s[quantile_str]
    df = df.with_columns(pl.Series('sum_sqres', list_sum_sqres[:-1]))
    
    best_picked_combos_dfs[quantile_str] = df

View the results:

In [105]:
for quantile_str, df in best_picked_combos_dfs.items():
    print(f'First fixed: {quantile_str}')
    print(f'Overall sum sqres: {best_picked_combos_r2s[quantile_str][-1]:.5f}')
    display(df)
    print('')

First fixed: q00
Overall sum sqres: 234.50419


depriv_quantile_min,less65,65-70,70-75,75-80,over80,sum_sqres
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.006,0.006,0.006,0.014,110.354891
"""q02""",0.0004,0.003,0.004,0.006,0.014,107.800573
"""q04""",0.0003,0.002,0.003,0.006,0.014,104.55857
"""q06""",0.0003,0.002,0.003,0.006,0.012,102.55868
"""q08""",0.0002,0.002,0.003,0.005,0.012,98.702951



First fixed: q02
Overall sum sqres: 234.50419


depriv_quantile_min,less65,65-70,70-75,75-80,over80,sum_sqres
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.006,0.006,0.006,0.014,110.354891
"""q02""",0.0004,0.003,0.004,0.006,0.014,107.800573
"""q04""",0.0003,0.002,0.003,0.006,0.014,104.55857
"""q06""",0.0003,0.002,0.003,0.006,0.012,102.55868
"""q08""",0.0002,0.002,0.003,0.005,0.012,98.702951



First fixed: q04
Overall sum sqres: 234.63866


depriv_quantile_min,less65,65-70,70-75,75-80,over80,sum_sqres
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.005,0.005,0.007,0.015,110.442113
"""q02""",0.0004,0.003,0.003,0.007,0.014,107.823731
"""q04""",0.0003,0.002,0.002,0.007,0.014,104.525757
"""q06""",0.0003,0.002,0.002,0.007,0.012,102.623322
"""q08""",0.0002,0.002,0.002,0.007,0.012,98.86709



First fixed: q06
Overall sum sqres: 234.50419


depriv_quantile_min,less65,65-70,70-75,75-80,over80,sum_sqres
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.006,0.006,0.006,0.014,110.354891
"""q02""",0.0004,0.003,0.004,0.006,0.014,107.800573
"""q04""",0.0003,0.002,0.003,0.006,0.014,104.55857
"""q06""",0.0003,0.002,0.003,0.006,0.012,102.55868
"""q08""",0.0002,0.002,0.003,0.005,0.012,98.702951



First fixed: q08
Overall sum sqres: 234.73615


depriv_quantile_min,less65,65-70,70-75,75-80,over80,sum_sqres
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.006,0.006,0.006,0.014,110.354891
"""q02""",0.0004,0.003,0.004,0.006,0.014,107.800573
"""q04""",0.0002,0.003,0.004,0.005,0.014,104.839407
"""q06""",0.0002,0.003,0.004,0.005,0.012,102.870443
"""q08""",0.0001,0.003,0.004,0.004,0.012,98.631977


Compare the overall r-squared scores for the five options:

In [107]:
pl.DataFrame(
    np.vstack((
        quantile_str_list,
        [round(best_picked_combos_r2s[quantile_str][-1], 3) for quantile_str in quantile_str_list]
    )).T,
    schema=['fixed_depriv_quantile_min', 'sum_sqres']
)

fixed_depriv_quantile_min,sum_sqres
str,str
"""q00""","""234.504"""
"""q02""","""234.504"""
"""q04""","""234.639"""
"""q06""","""234.504"""
"""q08""","""234.736"""


The lowest overall sum of square residuals is for the sets of coefficients that first fixed the 0.0-0.2, 0.2-0.4 and 0.6-0.8 deprivation quantiles.

Compare results with SSNAP-derived coefficients

Recalculate total fitness:

In [109]:
best_picked_combos_dfs['q00']

depriv_quantile_min,less65,65-70,70-75,75-80,over80,sum_sqres
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.006,0.006,0.006,0.014,110.354891
"""q02""",0.0004,0.003,0.004,0.006,0.014,107.800573
"""q04""",0.0003,0.002,0.003,0.006,0.014,104.55857
"""q06""",0.0003,0.002,0.003,0.006,0.012,102.55868
"""q08""",0.0002,0.002,0.003,0.005,0.012,98.702951


In [110]:
sum_sqres, rat, fitness, ratios_by_age, sum_sqres_by_depriv = eval_admissions(
    best_picked_combos['q00'],
    admissions_lists,
    x_lists,
    admissions_by_age
)

# Best R^2 from all genetic algorithm results:
fitness_best = df_best_gens['fitness'].min()
sum_sqres_best = df_best_gens['sum_sqres'].min()

print(f'New sum_sqres: {sum_sqres:.4f}')
print(f'Old sum_sqres: {sum_sqres_best:.4f}')

print(f'New fitness: {fitness:.4f}')
print(f'Old fitness: {fitness_best:.4f}')

New sum_sqres: 234.5042
Old sum_sqres: 237.4549
New fitness: 338.3078
Old fitness: 252.5461


Sum of square residuals has improved using these nudged coefficients, but fitness is much worse.

So it doesn't look likely that we'll find a good fit by looking at deprivation band alone.

### Test 4: Optimise age bands

can we pick out a few combos that are good for each age band, then put five sets together to get 25 coeffs that have good overall fitness?

The randomly-picked best fits seem to improve the "wrongness" ratio in the age bands much more than the overall sum of square residuals. So find ways to nudge the coefficients that result in better fits to the total national admissions for each age band.

In [111]:
dict_offsets_for_ages = {  # np.array([1e-4, 1e-3, 1e-3, 1e-3, 1e-3])
    'less65': 1e-4,
    '65': 1e-3,
    '70': 1e-3,
    '75': 1e-3,
    'over80': 1e-3,
}

All combos of offsets for ages:

In [112]:
combo_scales_for_ages = np.array([c for c in itertools.product([-2, -1, 0, 1, 2], repeat=5)])

In [113]:
len(combo_scales_for_ages)

3125

For a set of coeffs, apply these offsets and then recalculate fitness:

In [114]:
def wrong_rat(x_lists_here, coeffs, admissions_by_age):
    # Predictions for each age band across England:
    # Separate prediction for each deprivation quantile:
    # coeffs = individual[(i*5):(i*5)+5] #* np.array(coeffs_ssnap)
    yhat_list = (
        [(x_lists_here[i] * coeffs[i]).sum() for i in range(len(coeffs))]
    )
    predictions_by_age = sum(yhat_list)
    # Divide the admissions by age band by the observed values:
    predictions_by_age /= admissions_by_age_here
    return predictions_by_age

In [115]:
age_bands = ['less65', '65', '70', '75', 'over80']

dict_nudged_dfs = {}
for a, age_band in enumerate(age_bands):
    x_lists_here = [x_lists[i][a] for i in range(5)]  # only data for this age band
    admissions_by_age_here = admissions_by_age[a]
    
    age_cols = [c for c in df_best_gens.columns if f'_{age_band}_' in c]
    age_offset = dict_offsets_for_ages[age_band]
    combo_offsets_for_age = combo_scales_for_ages * age_offset
    
    start_coeffs = df_best_gens.sort('fitness')[0][age_cols].to_numpy().flatten()
    
    # df_nudge_here = pl.DataFrame(schema=age_cols + ['rat'])
    df_nudge_here = pl.DataFrame()
    # rats = []#[] for i in range(5)]
    
    for c, combo in enumerate(combo_offsets_for_age):
        coeffs_here = start_coeffs + np.array(combo)
        # Check that coeffs increase with deprivation:
        if all(np.diff(coeffs_here) <= 0.0) & all(coeffs_here > 0.0):
            rat = wrong_rat(x_lists_here, coeffs_here, admissions_by_age_here)
            abs_rat = abs(1.0 - rat)
            # rats.append(round(rat, 7))
            row = dict(zip(age_cols + ['rat', 'abs_rat'], list(coeffs_here) + [round(rat, 7), round(abs_rat, 7)]))
            # if c == 0:
            #     df_nudge_here = pl.DataFrame(row)
            # else:
            df_nudge_here = pl.concat((df_nudge_here, pl.DataFrame(row)))
        else:
            # Don't calculate results for here.
            pass
    # Store result:
    dict_nudged_dfs[age_band] = df_nudge_here

In [116]:
coeffs_ssnap

array([0.000408, 0.002644, 0.003735, 0.006005, 0.011558])

In [117]:
coeffs_ssnap_round = [0.0004, 0.003, 0.004, 0.006, 0.012]

In [118]:
i = 0
for key, df in dict_nudged_dfs.items():
    print(key)
    display(df.filter(df[f'age_{key}_q04'] == coeffs_ssnap_round[i]).sort('abs_rat')[:10])
    i += 1

less65


age_less65_q00,age_less65_q02,age_less65_q04,age_less65_q06,age_less65_q08,rat,abs_rat
f64,f64,f64,f64,f64,f64,f64
0.0006,0.0006,0.0004,0.0002,0.0002,0.9996408,0.0003592
0.0006,0.0006,0.0004,0.0003,0.0001,0.999514,0.000486
0.0007,0.0005,0.0004,0.0003,0.0001,1.0012324,0.0012324
0.0007,0.0005,0.0004,0.0002,0.0002,1.0013592,0.0013592
0.0007,0.0004,0.0004,0.0003,0.0002,0.9955844,0.0044156
0.0007,0.0004,0.0004,0.0004,0.0001,0.9954576,0.0045424
0.0006,0.0005,0.0004,0.0003,0.0002,0.993866,0.006134
0.0006,0.0005,0.0004,0.0004,0.0001,0.9937392,0.0062608
0.0007,0.0006,0.0004,0.0002,0.0001,1.0070071,0.0070071


65


age_65_q00,age_65_q02,age_65_q04,age_65_q06,age_65_q08,rat,abs_rat
f64,f64,f64,f64,f64,f64,f64
0.005,0.003,0.003,0.002,0.001,0.9978634,0.0021366
0.004,0.004,0.003,0.002,0.001,1.0051216,0.0051216
0.005,0.004,0.003,0.001,0.001,0.9833508,0.0166492
0.004,0.003,0.003,0.002,0.002,1.01735,0.01735
0.004,0.003,0.003,0.003,0.001,1.0196342,0.0196342
0.006,0.003,0.003,0.001,0.001,0.9760925,0.0239075
0.003,0.003,0.003,0.003,0.002,1.0391208,0.0391208
0.003,0.003,0.003,0.003,0.001,0.9581258,0.0418742
0.003,0.003,0.003,0.002,0.002,0.9558416,0.0441584


70


age_70_q00,age_70_q02,age_70_q04,age_70_q06,age_70_q08,rat,abs_rat
f64,f64,f64,f64,f64,f64,f64
0.006,0.006,0.004,0.003,0.001,0.9991764,0.0008236
0.006,0.006,0.004,0.002,0.002,0.9988826,0.0011174
0.005,0.004,0.004,0.004,0.002,0.9903172,0.0096828
0.005,0.004,0.004,0.003,0.003,0.9900235,0.0099765
0.004,0.004,0.004,0.004,0.003,1.0132933,0.0132933
0.006,0.005,0.004,0.003,0.002,1.0142795,0.0142795
0.006,0.005,0.004,0.004,0.001,1.0145733,0.0145733
0.005,0.005,0.004,0.004,0.001,0.9752141,0.0247859
0.005,0.005,0.004,0.003,0.002,0.9749203,0.0250797


75


age_75_q00,age_75_q02,age_75_q04,age_75_q06,age_75_q08,rat,abs_rat
f64,f64,f64,f64,f64,f64,f64
0.007,0.007,0.006,0.005,0.005,0.9995902,0.0004098
0.007,0.007,0.006,0.006,0.004,0.9992987,0.0007013
0.009,0.008,0.006,0.004,0.004,0.9970786,0.0029214
0.009,0.008,0.006,0.005,0.003,0.9967871,0.0032129
0.008,0.006,0.006,0.005,0.005,0.9941362,0.0058638
0.008,0.006,0.006,0.006,0.004,0.9938447,0.0061553
0.009,0.007,0.006,0.006,0.003,1.0072999,0.0072999
0.009,0.007,0.006,0.005,0.004,1.0075914,0.0075914
0.007,0.006,0.006,0.006,0.005,1.010103,0.010103


over80


age_over80_q00,age_over80_q02,age_over80_q04,age_over80_q06,age_over80_q08,rat,abs_rat
f64,f64,f64,f64,f64,f64,f64
0.015,0.014,0.012,0.011,0.008,1.0000637,0.0000637
0.014,0.012,0.012,0.011,0.01,0.9995308,0.0004692
0.015,0.014,0.012,0.01,0.009,1.0009259,0.0009259
0.014,0.012,0.012,0.012,0.009,0.9986685,0.0013315
0.013,0.013,0.012,0.012,0.009,1.0013433,0.0013433
0.016,0.013,0.012,0.01,0.009,0.9982512,0.0017488
0.013,0.013,0.012,0.011,0.01,1.0022055,0.0022055
0.016,0.012,0.012,0.012,0.008,1.0024256,0.0024256
0.016,0.013,0.012,0.011,0.008,0.9973889,0.0026111


In [119]:
i = 0
dict_nudged_dfs_ssnapq04 = {}
for key, df in dict_nudged_dfs.items():
    dict_nudged_dfs_ssnapq04[key] = df.filter(df[f'age_{key}_q04'] == coeffs_ssnap_round[i]).sort('abs_rat')
    i += 1


gather combos that are pretty good, check for increasing coeffs, calculate r-squared. Try various combos depending on resulting total r-squared by depriv quantile, e.g. if best coeffs all weighted heavily in one quantile and the answer is too high, then pick that quantile first to change to another set of coeffs.

..

Check all combos of the top 6 options. Picked because it runs a decent amount (7776 combos) but doesn't take forever to run.

In [120]:
# Every combo of the top ten indices:
index_combos = [c for c in itertools.product(range(6), repeat=5)]

In [121]:
fitness_keys = [
    'sum_sqres',
    'wrong_rat',
    'fitness',
    'wrong_rat_under65',
    'wrong_rat_65',
    'wrong_rat_70',
    'wrong_rat_75',
    'wrong_rat_over80',
    'sum_sqres_q00',
    'sum_sqres_q02',
    'sum_sqres_q04',
    'sum_sqres_q06',
    'sum_sqres_q08',
]

df_index_combos = pl.DataFrame()

for i, index_combo in enumerate(index_combos):
    print(f'{i+1:4d} out of {len(index_combos)}', end='\r')
    
    dict_inds = {
        'less65': index_combo[0],
        '65': index_combo[1],
        '70': index_combo[2],
        '75': index_combo[3],
        'over80': index_combo[4]
    }
    
    # Gather coeffs for these inds:
    all_coeff_cols = []
    all_coeffs = []
    for age_band, df in dict_nudged_dfs_ssnapq04.items():
        # 'less65', '65', '70', '75', 'over80'
        # Pick out the coeffs for this age band:
        coeff_cols_here = [c for c in df.columns if c.startswith('age')]
        all_coeff_cols += coeff_cols_here
        all_coeffs += list(df[dict_inds[age_band]][coeff_cols_here].to_numpy().flatten())
    # Order coeffs as usual:
    df_coeffs = pl.DataFrame(np.array(all_coeffs).reshape(1, len(all_coeffs)), schema=all_coeff_cols)
    df_coeffs = df_coeffs[coeff_names]
    coeffs = df_coeffs.to_numpy().flatten().reshape(5, 5)
    # print(coeffs)
    # print('\n')

    # Check that coeffs increase with deprivation:
    if np.all(np.diff(coeffs, axis=1) >= 0.0):
        # Calculate fitness:
        sum_sqres, rat, fitness, ratios_by_age, sum_sqres_by_depriv = eval_admissions(
            coeffs.flatten(),
            admissions_lists,
            x_lists,
            admissions_by_age
        )
        # Gather values for results:
        results_values = np.concatenate((coeffs.flatten(), [sum_sqres, rat, fitness], ratios_by_age, sum_sqres_by_depriv))
        df_coeffs = pl.DataFrame(results_values.reshape(1, len(results_values)), schema=coeff_names + fitness_keys)
        # Store result:
        df_index_combos = pl.concat((df_index_combos, df_coeffs))
    else:
        # Don't bother with coeffs that don't increase with deprivation.
        pass

In [122]:
df_index_combos.sort('fitness')

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0007,0.004,0.004,0.007,0.014,0.0004,0.004,0.004,0.007,0.012,0.0004,0.003,0.004,0.006,0.012,0.0003,0.002,0.004,0.005,0.011,0.0002,0.001,0.003,0.005,0.01,249.34486,1.023709,255.256694,0.995584,1.005122,1.013293,0.99959,0.999531,120.670912,107.931813,108.96863,103.869578,115.32156
0.0007,0.004,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.012,0.0004,0.003,0.004,0.006,0.012,0.0003,0.002,0.004,0.005,0.011,0.0002,0.002,0.003,0.005,0.01,246.495349,1.035938,255.353854,0.995584,1.01735,1.013293,0.99959,0.999531,120.670912,109.054005,108.96863,103.869578,107.901161
0.0006,0.004,0.004,0.007,0.014,0.0006,0.003,0.004,0.007,0.012,0.0004,0.003,0.004,0.006,0.012,0.0002,0.002,0.004,0.005,0.011,0.0002,0.002,0.003,0.005,0.01,248.227712,1.031881,256.141575,0.999641,1.01735,1.013293,0.99959,0.999531,112.578832,116.188715,108.96863,109.207891,107.901161
0.0006,0.004,0.004,0.008,0.014,0.0006,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0002,0.002,0.004,0.005,0.011,0.0002,0.002,0.003,0.005,0.01,247.776672,1.037335,257.027519,0.999641,1.01735,1.013293,0.994136,0.999531,113.949234,113.866873,108.96863,109.207891,107.901161
0.0007,0.004,0.004,0.007,0.014,0.0005,0.003,0.004,0.007,0.012,0.0004,0.003,0.004,0.006,0.012,0.0002,0.002,0.004,0.005,0.011,0.0002,0.002,0.003,0.005,0.01,248.930102,1.032881,257.115289,1.001359,1.01735,1.013293,0.99959,0.999531,120.670912,109.368827,108.96863,109.207891,107.901161
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.0007,0.006,0.006,0.009,0.015,0.0005,0.003,0.006,0.008,0.014,0.0004,0.003,0.004,0.006,0.012,0.0003,0.001,0.003,0.004,0.011,0.0001,0.001,0.001,0.004,0.008,330.92523,1.028949,340.50505,1.001232,0.976093,0.999176,0.997079,1.000064,163.23448,132.570292,108.96863,120.390711,197.289241
0.0007,0.005,0.006,0.009,0.015,0.0005,0.004,0.006,0.008,0.014,0.0004,0.003,0.004,0.006,0.012,0.0003,0.001,0.003,0.005,0.011,0.0001,0.001,0.001,0.003,0.008,335.13149,1.021982,342.498283,1.001232,0.983351,0.999176,0.996787,1.000064,154.250404,141.349918,108.96863,113.984555,208.982028
0.0006,0.005,0.006,0.009,0.015,0.0006,0.004,0.006,0.008,0.014,0.0004,0.003,0.004,0.006,0.012,0.0003,0.001,0.003,0.005,0.011,0.0001,0.001,0.001,0.003,0.008,335.535155,1.021235,342.660396,0.999514,0.983351,0.999176,0.996787,1.000064,136.54163,159.374072,108.96863,113.984555,208.982028


Best fitnesses worse than the start value!

In [123]:
df_index_combos.sort('fitness')[0][coeff_names].to_numpy().reshape(5, 5)

array([[0.0007, 0.004 , 0.004 , 0.007 , 0.014 ],
       [0.0004, 0.004 , 0.004 , 0.007 , 0.012 ],
       [0.0004, 0.003 , 0.004 , 0.006 , 0.012 ],
       [0.0003, 0.002 , 0.004 , 0.005 , 0.011 ],
       [0.0002, 0.001 , 0.003 , 0.005 , 0.01  ]])

In [124]:
coeffs_ssnap

array([0.000408, 0.002644, 0.003735, 0.006005, 0.011558])

In [125]:
df_best_gens.sort('fitness')[0]

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed74""",22.0,0.0005,0.004,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.191107,1.055834,252.546109,0.980725,1.01735,1.013293,0.99959,0.994494,111.087199,107.849722,108.96863,104.473782,102.232696


## Pair-wise nudging

walk around the starting coeffs.

Try all pairs of nudges in each row and column. Pick the one that has best effect on fitness. Then find best pair to match whatever has already changed. Keep iterating until no more improvements can be made.

Set up pair indices.

In [126]:
row_col_inds = [c for c in itertools.product(range(5), repeat=2)]

row_col_inds

[(0, 0),
 (0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (1, 0),
 (1, 1),
 (1, 2),
 (1, 3),
 (1, 4),
 (2, 0),
 (2, 1),
 (2, 2),
 (2, 3),
 (2, 4),
 (3, 0),
 (3, 1),
 (3, 2),
 (3, 3),
 (3, 4),
 (4, 0),
 (4, 1),
 (4, 2),
 (4, 3),
 (4, 4)]

In [127]:
def build_nudge_df(start_coeffs, first_pass=False):
    """
    """
    def convert_to_dataframe(coeffs_here, nudged_cols):
        """
        """
        # Convert to dataframe:
        data_here = coeffs_here.flatten()
        df_here = pl.DataFrame(np.array(data_here).reshape(1, len(data_here)), schema=nudged_cols)
        return df_here
    
    nudged_cols = coeff_names
    
    df_nudges = pl.DataFrame()
    
    # coeffs_here = np.array([s for s in start_coeffs]).reshape(5, 5)
    # # Store result:
    # df_here = convert_to_dataframe(coeffs_here, nudged_cols)
    # df_nudges = pl.concat((df_nudges, df_here))
    if first_pass:
        # Force two coeffs to change.
          m_pairs_available = [(1.0, -1.0), (-1.0, 1.0)]
    else:
        # Only one coeff has to change, but two are allowed to.
        m_pairs_available = [(1.0, -1.0), (-1.0, 1.0), (1.0, 0.0), (0.0, 1.0), (-1.0, 0.0), (0.0, -1.0)]
    
    for r, (row_a, col_a) in enumerate(row_col_inds[:3]):
        step_a = 1e-4 if col_a == 0 else 1e-3
        # Compare with each other point in the same row:
        for col_b in range(5):
            if col_b != col_a:
                step_b = 1e-4 if col_b == 0 else 1e-3
                for m_pairs in m_pairs_available:
                    # Nudge coefficients:
                    coeffs_here = np.array([s for s in start_coeffs]).reshape(5, 5)
                    coeffs_here[row_a, col_a] += m_pairs[0]*step_a
                    coeffs_here[row_a, col_b] += m_pairs[1]*step_b
                    # Store result:
                    df_here = convert_to_dataframe(coeffs_here, nudged_cols)
                    df_nudges = pl.concat((df_nudges, df_here))
        for row_b in range(5):
            if row_b != row_a:
                # Same step size.
                for m_pairs in m_pairs_available:
                    # Nudge coefficients:
                    coeffs_here = np.array([s for s in start_coeffs]).reshape(5, 5)
                    coeffs_here[row_a, col_a] += m_pairs[0]*step_a
                    coeffs_here[row_b, col_a] += m_pairs[1]*step_a
        
                    # Store result:
                    df_here = convert_to_dataframe(coeffs_here, nudged_cols)
                    df_nudges = pl.concat((df_nudges, df_here))
    # Remove repeats
    # (from comparing A then B and also B then A):
    df_nudges = df_nudges.unique()
    return df_nudges

In [128]:
def find_nudged_fitnesses(df_nudges):
    df_nudges_fitness = pl.DataFrame()
    
    fitness_keys = [
        'sum_sqres',
        'wrong_rat',
        'fitness',
        'wrong_rat_under65',
        'wrong_rat_65',
        'wrong_rat_70',
        'wrong_rat_75',
        'wrong_rat_over80',
        'sum_sqres_q00',
        'sum_sqres_q02',
        'sum_sqres_q04',
        'sum_sqres_q06',
        'sum_sqres_q08',
    ]
    for i in range(len(df_nudges)):
        coeffs = df_nudges[i][coeff_names].to_numpy().flatten()
        
        if np.all(np.diff(coeffs.reshape(5, 5), axis=1) >= 0.0) & np.all(np.diff(coeffs.reshape(5, 5), axis=0) <= 0.0):
            sum_sqres, rat, fitness, ratios_by_age, sum_sqres_by_depriv = eval_admissions(
                coeffs,
                admissions_lists,
                x_lists,
                admissions_by_age
            )
        else:        
            # Overwrite results if coeffs invalid:
            sum_sqres = np.nan
            rat = np.nan
            fitness = np.nan
            ratios_by_age = [np.nan] * 5
            sum_sqres_by_depriv = [np.nan] * 5
    
        # Gather values for results:
        results_values = np.concatenate((coeffs.flatten(), [sum_sqres, rat, fitness], ratios_by_age, sum_sqres_by_depriv))
        df_coeffs = pl.DataFrame(results_values.reshape(1, len(results_values)), schema=coeff_names + fitness_keys)
        # Store result:
        df_nudges_fitness = pl.concat((df_nudges_fitness, df_coeffs))
    return df_nudges_fitness

In [129]:
dict_nudged_results = {}

for d in df_best_gens['dir']:
    
    start_coeffs = df_best_gens.filter(df_best_gens['dir'] == d)[coeff_names].to_numpy().flatten()
    df_nudge_history = find_nudged_fitnesses(pl.DataFrame(start_coeffs.reshape(1, len(start_coeffs)), schema=coeff_names))
    
    keep_going_please = True
    k = 0
    while keep_going_please:
        first_pass = False if k > 0 else True
        df_nudges = build_nudge_df(start_coeffs, first_pass)
        df_nudges = find_nudged_fitnesses(df_nudges)
        # Pick out the best one:
        df_nudges = df_nudges.sort('fitness')[0]  # CONSIDER not doing fitness, separate r2 or wrong rat depending on whether row or col changed?
        start_coeffs = df_nudges[coeff_names].to_numpy().flatten()
        df_nudge_history = pl.concat((df_nudge_history, df_nudges))
        k += 1
        if (k > 2) & (len(df_nudge_history) != len(df_nudge_history.unique())):
            keep_going_please = False
        elif k > 100:
            keep_going_please = False
            print('Cutting off after 100 iterations.')

    # Store results:
    dict_nudged_results[d] = df_nudge_history

In [130]:
df_best_nudged = pl.DataFrame()

for d, df in dict_nudged_results.items():
    df_best_nudged = pl.concat((df_best_nudged, df.filter(df['fitness'] == df['fitness'].min())[0]))

In [131]:
with pl.Config(set_tbl_rows=100):
    display(df_best_nudged.sort('fitness'))

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0005,0.005,0.005,0.007,0.013,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,237.458594,1.031902,245.034106,0.980725,1.000036,0.990024,0.99959,1.002206,110.509875,107.849722,106.276266,103.905822,102.232696
0.0005,0.005,0.005,0.007,0.013,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,237.458594,1.031902,245.034106,0.980725,1.000036,0.990024,0.99959,1.002206,110.509875,107.849722,106.276266,103.905822,102.232696
0.0005,0.005,0.005,0.007,0.013,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,237.458594,1.031902,245.034106,0.980725,1.000036,0.990024,0.99959,1.002206,110.509875,107.849722,106.276266,103.905822,102.232696
0.0005,0.005,0.005,0.007,0.013,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,237.458594,1.031902,245.034106,0.980725,1.000036,0.990024,0.99959,1.002206,110.509875,107.849722,106.276266,103.905822,102.232696
0.0005,0.005,0.005,0.007,0.013,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,237.458594,1.031902,245.034106,0.980725,1.000036,0.990024,0.99959,1.002206,110.509875,107.849722,106.276266,103.905822,102.232696
0.0005,0.005,0.005,0.007,0.013,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,237.458594,1.031902,245.034106,0.980725,1.000036,0.990024,0.99959,1.002206,110.509875,107.849722,106.276266,103.905822,102.232696
0.0005,0.005,0.005,0.007,0.013,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,237.458594,1.031902,245.034106,0.980725,1.000036,0.990024,0.99959,1.002206,110.509875,107.849722,106.276266,103.905822,102.232696
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.012,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,238.365332,1.030166,245.555895,0.980725,1.000036,0.990024,0.99959,0.999531,111.277433,109.054005,106.276266,103.905822,102.232696
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.011,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,237.99558,1.033271,245.91396,0.980725,1.000036,0.990024,0.99959,0.996426,111.277433,107.849722,106.675784,103.905822,102.232696


In [132]:
df_best_nudged.sort('fitness')[0][coeff_names].to_numpy().flatten().reshape(5, 5)

array([[0.0005, 0.005 , 0.005 , 0.007 , 0.013 ],
       [0.0004, 0.003 , 0.004 , 0.007 , 0.013 ],
       [0.0004, 0.002 , 0.004 , 0.006 , 0.012 ],
       [0.0004, 0.002 , 0.003 , 0.005 , 0.011 ],
       [0.0003, 0.002 , 0.003 , 0.005 , 0.01  ]])

In [133]:
pl.concat((df_best_nudged.mean(), df_best_nudged.std(), 100.0*df_best_nudged.std() / df_best_nudged.mean()))

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.000479,0.00474,0.00477,0.00735,0.0143,0.000406,0.00313,0.004,0.00665,0.01292,0.000398,0.00215,0.00399,0.00595,0.01176,0.000393,0.002,0.00325,0.00515,0.0103,0.000324,0.00198,0.003,0.00497,0.00999,238.859453,1.043677,249.290567,0.979477,1.003187,0.996055,1.000683,0.99845,111.223638,108.152613,106.616943,105.660788,102.230866
0.000048,0.000441,0.000489,0.000575,0.00102,0.000024,0.000338,0.0,0.000479,0.000367,0.000014,0.000359,0.0001,0.000219,0.000429,0.000026,0.0,0.000435,0.000359,0.000461,0.000043,0.000141,0.0,0.000171,0.000174,0.894136,0.009965,2.388647,0.003975,0.006029,0.010228,0.009497,0.004516,0.583905,0.581558,0.829024,1.723623,0.356809
9.972522,9.300506,10.259585,7.825295,7.132881,5.878898,10.798648,0.0,7.208609,2.843303,3.535309,16.691641,2.506266,3.681394,3.649955,6.524997,0.0,13.390589,6.968355,4.471516,13.247984,7.106328,0.0,3.44963,1.739612,0.374336,0.95475,0.958178,0.40583,0.600998,1.026829,0.949087,0.452256,0.524983,0.53772,0.777573,1.63128,0.349022


In [134]:
with pl.Config(set_tbl_rows=12):
    for d, df in dict_nudged_results.items():
        display(df_best_gens.filter(df_best_gens['dir'] == d))
        display(df)
        print(df_best_gens.filter(df_best_gens['dir'] == d)[coeff_names].to_numpy().flatten().reshape(5, 5))
        print('')
        print(df.filter(df['fitness'] == df['fitness'].min())[0][coeff_names].to_numpy().flatten().reshape(5, 5))
        print('\n'*2)

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed00""",33.0,0.0005,0.003,0.005,0.007,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.006,0.011,0.0003,0.002,0.003,0.005,0.01,240.103163,1.140725,273.891734,0.980725,0.955842,1.052652,1.010103,1.014536,111.220039,108.40745,108.96863,105.838415,102.232696


age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0005,0.003,0.005,0.007,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.006,0.011,0.0003,0.002,0.003,0.005,0.01,240.103163,1.140725,273.891734,0.980725,0.955842,1.052652,1.010103,1.014536,111.220039,108.40745,108.96863,105.838415,102.232696
0.0005,0.004,0.004,0.007,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.006,0.011,0.0003,0.002,0.003,0.005,0.01,240.041658,1.074558,257.938603,0.980725,1.01735,1.013293,1.010103,1.014536,111.087199,108.40745,108.96863,105.838415,102.232696
0.0006,0.004,0.004,0.007,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0003,0.002,0.004,0.006,0.011,0.0003,0.002,0.003,0.005,0.01,239.488729,1.067064,255.549917,0.988218,1.01735,1.013293,1.010103,1.014536,112.578832,108.40745,108.96863,102.970852,102.232696
0.0006,0.004,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0003,0.002,0.004,0.006,0.011,0.0003,0.002,0.003,0.005,0.01,239.080657,1.054734,252.166432,0.988218,1.01735,1.013293,1.010103,1.002206,111.708112,108.40745,108.96863,102.970852,102.232696
0.0007,0.004,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.013,0.0003,0.003,0.004,0.006,0.012,0.0003,0.002,0.004,0.006,0.011,0.0003,0.002,0.003,0.005,0.01,240.069885,1.049826,252.03151,0.993126,1.01735,1.013293,1.010103,1.002206,117.371022,108.40745,105.123247,102.970852,102.232696
0.0006,0.004,0.004,0.007,0.013,0.0005,0.003,0.004,0.006,0.013,0.0003,0.003,0.004,0.006,0.012,0.0003,0.002,0.004,0.006,0.011,0.0003,0.002,0.003,0.005,0.01,237.923789,1.051544,250.187335,0.991408,1.01735,1.013293,1.010103,1.002206,111.708112,109.65222,105.123247,102.970852,102.232696
0.0006,0.004,0.005,0.007,0.013,0.0005,0.003,0.004,0.006,0.013,0.0003,0.003,0.004,0.006,0.012,0.0003,0.002,0.003,0.006,0.011,0.0003,0.002,0.003,0.005,0.01,239.010104,1.048227,250.536904,0.991408,1.01735,0.990024,1.010103,1.002206,112.561119,109.65222,105.123247,104.545655,102.232696
0.0006,0.005,0.005,0.007,0.013,0.0005,0.003,0.004,0.006,0.013,0.0003,0.002,0.004,0.006,0.012,0.0003,0.002,0.003,0.006,0.011,0.0003,0.002,0.003,0.005,0.01,240.553762,1.030913,247.989952,0.991408,1.000036,0.990024,1.010103,1.002206,114.965752,109.65222,106.038015,104.545655,102.232696
0.0005,0.005,0.005,0.007,0.013,0.0005,0.003,0.004,0.006,0.013,0.0004,0.002,0.004,0.006,0.012,0.0003,0.002,0.003,0.006,0.011,0.0003,0.002,0.003,0.005,0.01,238.56237,1.035821,247.107896,0.9865,1.000036,0.990024,1.010103,1.002206,110.509875,109.65222,106.276266,104.545655,102.232696


[[0.0005 0.003  0.005  0.007  0.014 ]
 [0.0004 0.003  0.004  0.006  0.013 ]
 [0.0004 0.003  0.004  0.006  0.012 ]
 [0.0004 0.002  0.004  0.006  0.011 ]
 [0.0003 0.002  0.003  0.005  0.01  ]]

[[0.0005 0.005  0.005  0.007  0.013 ]
 [0.0005 0.003  0.004  0.006  0.013 ]
 [0.0004 0.002  0.004  0.006  0.012 ]
 [0.0003 0.002  0.003  0.006  0.011 ]
 [0.0003 0.002  0.003  0.005  0.01  ]]





dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed01""",33.0,0.0004,0.003,0.005,0.008,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,242.540903,1.137745,275.949636,0.92776,0.955842,0.990024,0.994136,0.994494,114.648926,108.40745,108.96863,107.721177,102.232696


age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0004,0.003,0.005,0.008,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,242.540903,1.137745,275.949636,0.92776,0.955842,0.990024,0.994136,0.994494,114.648926,108.40745,108.96863,107.721177,102.232696
0.0005,0.003,0.005,0.008,0.013,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,241.178257,1.09711,264.599162,0.980725,0.955842,0.990024,0.994136,0.982163,111.737362,108.40745,108.96863,107.721177,102.232696
0.0005,0.004,0.005,0.008,0.013,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,240.666987,1.070302,257.586352,0.980725,1.01735,0.990024,0.994136,0.982163,110.629494,108.40745,108.96863,107.721177,102.232696
0.0005,0.005,0.005,0.008,0.013,0.0004,0.003,0.004,0.006,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.637069,1.052988,252.334845,0.980725,1.000036,0.990024,0.994136,0.982163,111.012291,108.40745,106.276266,107.721177,102.232696
0.0005,0.005,0.005,0.008,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,240.385844,1.040657,250.159156,0.980725,1.000036,0.990024,0.994136,0.994494,112.619528,108.40745,106.276266,107.721177,102.232696
0.0005,0.004,0.005,0.008,0.014,0.0004,0.004,0.004,0.006,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.331576,1.047915,250.799162,0.980725,1.007294,0.990024,0.994136,0.994494,110.922311,107.823111,106.276266,107.721177,102.232696
0.0005,0.004,0.004,0.008,0.014,0.0004,0.004,0.004,0.006,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,237.756318,1.051232,249.937007,0.980725,1.007294,1.013293,0.994136,0.994494,110.640424,107.823111,106.276266,104.473782,102.232696
0.0005,0.004,0.004,0.008,0.015,0.0004,0.004,0.004,0.006,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,237.903712,1.052551,250.405743,0.980725,1.007294,1.013293,0.994136,1.006825,110.956807,107.823111,106.276266,104.473782,102.232696
0.0005,0.004,0.004,0.008,0.014,0.0004,0.004,0.004,0.006,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,237.756318,1.051232,249.937007,0.980725,1.007294,1.013293,0.994136,0.994494,110.640424,107.823111,106.276266,104.473782,102.232696


...

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0005,0.003,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,240.257537,1.113647,267.56215,0.980725,1.039121,0.950664,0.99959,0.994494,113.136822,107.849722,108.96863,104.721032,102.232696
0.0005,0.003,0.005,0.007,0.013,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,240.234294,1.086619,261.043111,0.980725,1.039121,0.990024,0.99959,0.982163,113.087456,107.849722,108.96863,104.721032,102.232696
0.0005,0.004,0.005,0.007,0.013,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,240.616094,1.064848,256.219572,0.980725,1.01735,0.990024,0.99959,0.982163,111.063064,107.849722,108.96863,107.721177,102.232696
0.0005,0.005,0.005,0.007,0.013,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.152704,1.047534,250.520486,0.980725,1.000036,0.990024,0.99959,0.982163,110.509875,107.849722,106.276266,107.721177,102.232696
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.508351,1.035203,247.939719,0.980725,1.000036,0.990024,0.99959,0.994494,111.277433,107.849722,106.276266,107.721177,102.232696
0.0005,0.005,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,240.437787,1.036522,249.219007,0.980725,1.000036,0.990024,0.99959,1.006825,113.264056,107.849722,106.276266,107.721177,102.232696
0.0005,0.005,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.508351,1.035203,247.939719,0.980725,1.000036,0.990024,0.99959,0.994494,111.277433,107.849722,106.276266,107.721177,102.232696


[[0.0005 0.003  0.004  0.007  0.014 ]
 [0.0004 0.003  0.004  0.007  0.013 ]
 [0.0004 0.003  0.004  0.006  0.012 ]
 [0.0004 0.003  0.003  0.005  0.01  ]
 [0.0003 0.002  0.003  0.005  0.01  ]]

[[0.0005 0.005  0.005  0.007  0.014 ]
 [0.0004 0.003  0.004  0.007  0.013 ]
 [0.0004 0.002  0.004  0.006  0.012 ]
 [0.0004 0.002  0.003  0.005  0.01  ]
 [0.0003 0.002  0.003  0.005  0.01  ]]





dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed99""",19.0,0.0005,0.003,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0003,0.003,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,240.233255,1.123077,269.800397,0.935253,1.039121,1.013293,0.99959,0.994494,113.136822,107.849722,108.96863,104.665312,102.232696


age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0005,0.003,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0003,0.003,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,240.233255,1.123077,269.800397,0.935253,1.039121,1.013293,0.99959,0.994494,113.136822,107.849722,108.96863,104.665312,102.232696
0.0006,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0003,0.003,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.739897,1.082442,259.504634,0.988218,1.039121,1.013293,0.99959,0.982163,112.085422,107.849722,108.96863,104.665312,102.232696
0.0006,0.004,0.004,0.007,0.013,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0003,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,241.194112,1.060672,255.827744,0.988218,1.01735,1.013293,0.99959,0.982163,111.708112,107.849722,108.96863,108.345043,102.232696
0.0006,0.004,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0003,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,241.598614,1.048341,253.277695,0.988218,1.01735,1.013293,0.99959,0.994494,112.578832,107.849722,108.96863,108.345043,102.232696
0.0005,0.004,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.191107,1.055834,252.546109,0.980725,1.01735,1.013293,0.99959,0.994494,111.087199,107.849722,108.96863,104.473782,102.232696
0.0005,0.004,0.004,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,238.942044,1.057153,252.598301,0.980725,1.01735,1.013293,0.99959,1.006825,110.549901,107.849722,108.96863,104.473782,102.232696
0.0005,0.004,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.191107,1.055834,252.546109,0.980725,1.01735,1.013293,0.99959,0.994494,111.087199,107.849722,108.96863,104.473782,102.232696


[[0.0005 0.003  0.004  0.007  0.014 ]
 [0.0004 0.003  0.004  0.007  0.013 ]
 [0.0004 0.003  0.004  0.006  0.012 ]
 [0.0003 0.003  0.004  0.005  0.01  ]
 [0.0003 0.002  0.003  0.005  0.01  ]]

[[0.0005 0.004  0.004  0.007  0.014 ]
 [0.0004 0.003  0.004  0.007  0.013 ]
 [0.0004 0.003  0.004  0.006  0.012 ]
 [0.0004 0.002  0.004  0.005  0.01  ]
 [0.0003 0.002  0.003  0.005  0.01  ]]





In [135]:
df_best_gens.sort('fitness')[0][coeff_names].to_numpy().flatten().reshape(5, 5)

array([[0.0005, 0.004 , 0.004 , 0.007 , 0.014 ],
       [0.0004, 0.003 , 0.004 , 0.007 , 0.013 ],
       [0.0004, 0.003 , 0.004 , 0.006 , 0.012 ],
       [0.0004, 0.002 , 0.004 , 0.005 , 0.01  ],
       [0.0003, 0.002 , 0.003 , 0.005 , 0.01  ]])

In [136]:
df_nudge_history.sort('fitness')[0][coeff_names].to_numpy().flatten().reshape(5, 5)

array([[0.0005, 0.004 , 0.004 , 0.007 , 0.014 ],
       [0.0004, 0.003 , 0.004 , 0.007 , 0.013 ],
       [0.0004, 0.003 , 0.004 , 0.006 , 0.012 ],
       [0.0004, 0.002 , 0.004 , 0.005 , 0.01  ],
       [0.0003, 0.002 , 0.003 , 0.005 , 0.01  ]])

In [137]:
df_best_gens.sort('fitness')[0]

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed74""",22.0,0.0005,0.004,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.191107,1.055834,252.546109,0.980725,1.01735,1.013293,0.99959,0.994494,111.087199,107.849722,108.96863,104.473782,102.232696


In [138]:
start_coeffs.reshape(5, 5)

array([[0.0005, 0.004 , 0.004 , 0.007 , 0.014 ],
       [0.0004, 0.003 , 0.004 , 0.007 , 0.013 ],
       [0.0004, 0.003 , 0.004 , 0.006 , 0.012 ],
       [0.0004, 0.002 , 0.004 , 0.005 , 0.01  ],
       [0.0003, 0.002 , 0.003 , 0.005 , 0.01  ]])

In [139]:
df_nudges#.unique()

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,sum_sqres,wrong_rat,fitness,wrong_rat_under65,wrong_rat_65,wrong_rat_70,wrong_rat_75,wrong_rat_over80,sum_sqres_q00,sum_sqres_q02,sum_sqres_q04,sum_sqres_q06,sum_sqres_q08
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0005,0.004,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.191107,1.055834,252.546109,0.980725,1.01735,1.013293,0.99959,0.994494,111.087199,107.849722,108.96863,104.473782,102.232696
